# Tome 15, Chapter 5: Random groups (Python companion)

Sandpile groups computed by Smith normal form, the corank chain of Lemma 4.3 checked against brute force, the moments of Theorem 4.2 checked by simulation, Euler's identities, and real sandpile groups of random graphs against the law of Theorem 4.4. The computer-science application is **exact linear algebra over $\mathbb Z$, $\mathbb Z/p^k$ and $\mathbb F_q$**: Gaussian elimination with pivoting by valuation, and exact rational inverses to find the exponent of a finite abelian group. Exercises first, solutions at the end. Requires `numpy` only.

In [1]:
import math, itertools, numpy as np
from fractions import Fraction
rng = np.random.default_rng(5)

## Exercises

**P5.1 (★) Sandpile groups by hand and by machine.** Write a Smith normal form for integer matrices and compute the sandpile groups of the cycles $C_m$, the complete graphs $K_N$ (Exercises 4.5 and 4.6) and the Petersen graph.

**P5.2 (★) The chain against brute force.** Enumerate all symmetric $n\times n$ matrices over $\mathbb F_2$ ($n\le4$) and $\mathbb F_3$ ($n\le3$), tabulate their coranks, and compare with the distribution of the chain of Lemma 4.3 started at $0$.

**P5.3 (★★) Moments by simulation.** Over $\mathbb Z/9$, sample uniform symmetric $6\times6$ matrices, compute the cokernels, and estimate $\mathbb E\,\#\mathrm{Sur}(\operatorname{cok}M,G)$ for $G=\mathbb Z/3$, $\mathbb Z/9$ and $(\mathbb Z/3)^2$. Compare with the exact values of Theorem 4.2. Do the same for non-symmetric matrices (Exercise 4.4).

**P5.4 (★) Euler.** Check the two identities in the proof of Theorem 4.4 coefficient by coefficient up to $x^{40}$, and compute the limit laws for $q=2,3,5$.

**P5.5 (★★) Symmetric against non-symmetric.** Simulate the corank of uniform $n\times n$ matrices over $\mathbb F_3$ with and without symmetry, and compare the tails: $q^{-k^2/2}$ against $q^{-k^2}$.

**P5.6 (★★) Real sandpiles.** For $G(n,\frac12)$, compute the $3$-rank of the sandpile group (as in Figure 4.1) and the proportion of graphs whose sandpile group is cyclic, and compare with Remark 4.5.

## Solutions

**P5.1.** The Smith normal form follows the proof in Tome 2: move a non-zero entry of smallest absolute value to the corner, reduce its row and column by Euclidean division, repeat until the corner divides everything, then recurse. Python integers are exact, so there is no overflow.

In [2]:
def smith(A):
    A = [list(map(int, r)) for r in A]; m, n = len(A), len(A[0]); d = []
    for t in range(min(m, n)):
        while True:
            nz = [(abs(A[i][j]), i, j) for i in range(t, m) for j in range(t, n) if A[i][j]]
            if not nz: return d
            _, i, j = min(nz)
            A[t], A[i] = A[i], A[t]
            for r in A: r[t], r[j] = r[j], r[t]
            a = A[t][t]; done = True
            for i in range(t + 1, m):
                q = A[i][t] // a
                if q: A[i] = [x - q * y for x, y in zip(A[i], A[t])]
                if A[i][t]: done = False
            for j in range(t + 1, n):
                q = A[t][j] // a
                if q:
                    for r in A: r[j] -= q * r[t]
                if A[t][j]: done = False
            if done:
                bad = [(i, j) for i in range(t + 1, m) for j in range(t + 1, n) if A[i][j] % a]
                if not bad: break
                i, _ = bad[0]; A[t] = [x + y for x, y in zip(A[t], A[i])]
        d.append(abs(A[t][t]))
    return d

def reduced_laplacian(n, edges):
    L = [[0] * n for _ in range(n)]
    for u, v in edges:
        L[u][u] += 1; L[v][v] += 1; L[u][v] -= 1; L[v][u] -= 1
    return [r[1:] for r in L[1:]]

def group(d):
    f = [x for x in d if x != 1]
    return ' x '.join(f'Z/{x}' for x in f) if f else 'trivial'

for m in range(3, 9):
    print(f'C_{m}:', group(smith(reduced_laplacian(m, [(i, (i + 1) % m) for i in range(m)]))))
for N in range(3, 8):
    print(f'K_{N}:', group(smith(reduced_laplacian(N, list(itertools.combinations(range(N), 2))))))
pet = [(i, (i + 1) % 5) for i in range(5)] + [(i, i + 5) for i in range(5)] + [(5 + i, 5 + (i + 2) % 5) for i in range(5)]
d = smith(reduced_laplacian(10, pet)); print('Petersen:', group(d), ' order', math.prod(d))

C_3: Z/3
C_4: Z/4
C_5: Z/5
C_6: Z/6
C_7: Z/7
C_8: Z/8
K_3: Z/3
K_4: Z/4 x Z/4
K_5: Z/5 x Z/5 x Z/5
K_6: Z/6 x Z/6 x Z/6 x Z/6
K_7: Z/7 x Z/7 x Z/7 x Z/7 x Z/7
Petersen: Z/2 x Z/10 x Z/10 x Z/10  order 2000


The cycles give $\mathbb Z/m$ and the complete graphs $(\mathbb Z/N)^{N-2}$, as in the exercises. The Petersen graph has $2000$ spanning trees, and its sandpile group is $\mathbb Z/2\times(\mathbb Z/10)^3$: not cyclic.

**P5.2.** A symmetric matrix is determined by its upper triangle, $n(n+1)/2$ entries. Rank over $\mathbb F_q$ by Gaussian elimination; the chain distribution by iterating the transition probabilities with exact fractions.

In [3]:
def rank_mod(A, q):
    A = [[x % q for x in r] for r in A]; r = 0; rows = len(A); cols = len(A[0]) if A else 0
    for c in range(cols):
        piv = next((i for i in range(r, rows) if A[i][c]), None)
        if piv is None: continue
        A[r], A[piv] = A[piv], A[r]
        inv = pow(A[r][c], -1, q); A[r] = [x * inv % q for x in A[r]]
        for i in range(rows):
            if i != r and A[i][c]:
                f = A[i][c]; A[i] = [(x - f * y) % q for x, y in zip(A[i], A[r])]
        r += 1
    return r

def brute(n, q):
    cnt = {}; pos = [(i, j) for i in range(n) for j in range(i, n)]
    for vals in itertools.product(range(q), repeat=len(pos)):
        A = [[0] * n for _ in range(n)]
        for (i, j), v in zip(pos, vals): A[i][j] = A[j][i] = v
        k = n - rank_mod(A, q); cnt[k] = cnt.get(k, 0) + 1
    tot = q ** len(pos)
    return {k: Fraction(c, tot) for k, c in sorted(cnt.items())}

def chain(n, q):
    dist = {0: Fraction(1)}
    for _ in range(n):
        new = {}
        for k, pr in dist.items():
            Q = Fraction(1, q ** k)
            for kk, t in ((k - 1, 1 - Q), (k, Q * (1 - Fraction(1, q))), (k + 1, Q / q)):
                if t: new[kk] = new.get(kk, 0) + pr * t
        dist = new
    return dict(sorted(dist.items()))

for q, nmax in ((2, 4), (3, 3)):
    for n in range(1, nmax + 1):
        b, c = brute(n, q), chain(n, q)
        print(f'q={q} n={n}:', 'agree' if b == c else 'DIFFER', {k: str(v) for k, v in b.items()})

q=2 n=1: agree {0: '1/2', 1: '1/2'}
q=2 n=2: agree {0: '1/2', 1: '3/8', 2: '1/8'}
q=2 n=3: agree {0: '7/16', 1: '7/16', 2: '7/64', 3: '1/64'}
q=2 n=4: agree {0: '7/16', 1: '105/256', 2: '35/256', 3: '15/1024', 4: '1/1024'}
q=3 n=1: agree {0: '2/3', 1: '1/3'}
q=3 n=2: agree {0: '2/3', 1: '8/27', 2: '1/27'}
q=3 n=3: agree {0: '52/81', 1: '26/81', 2: '26/729', 3: '1/729'}


The chain reproduces the brute-force counts exactly, fraction for fraction.

**P5.3.** Over $R=\mathbb Z/p^k$ we diagonalise by pivoting on an entry of smallest $p$-adic valuation $v$: it is $p^v$ times a unit, and every other entry of the remaining matrix is divisible by $p^v$, so its row and column can be cleared. The cokernel is $\bigoplus\mathbb Z/p^{v_i}$ (with $v_i=k$ for zero diagonal entries). Then $\#\mathrm{Hom}(\bigoplus\mathbb Z/p^{a_i},\mathbb Z/p^b)=\prod_ip^{\min(a_i,b)}$, and a homomorphism to $\mathbb Z/9$ is onto unless it lands in $3\mathbb Z/9\cong\mathbb Z/3$; a homomorphism to $(\mathbb Z/3)^2$ factors through the $3$-rank $K$, and there are $(3^K-1)(3^K-3)$ onto ones.

In [4]:
def val(x, p, k):
    if x == 0: return k
    v = 0
    while x % p == 0: x //= p; v += 1
    return v

def cok_exponents(A, p, k):
    mod = p ** k; A = [[x % mod for x in r] for r in A]; n = len(A); ex = []
    rows, cols = list(range(n)), list(range(n))
    while rows:
        v, i, j = min((val(A[i][j], p, k), i, j) for i in rows for j in cols)
        ex.append(v)
        if v == k: ex += [k] * (len(rows) - 1); break
        u = A[i][j] // p ** v; inv = pow(u, -1, mod)
        for i2 in rows:
            if i2 != i and A[i2][j]:
                f = (A[i2][j] // p ** v) * inv % mod
                A[i2] = [(x - f * y) % mod for x, y in zip(A[i2], A[i])]
        rows.remove(i); cols.remove(j)
    return [e for e in ex if e > 0]

def homs(ex, p, b): return math.prod(p ** min(a, b) for a in ex)
def moments(ex, p=3):
    K = len(ex)
    return {'Z/3': homs(ex, p, 1) - 1, 'Z/9': homs(ex, p, 2) - homs(ex, p, 1), '(Z/3)^2': (p**K - 1) * (p**K - p)}

p, k, n, runs = 3, 2, 6, 20000
exact_sym = {'Z/3': 1 - p**-n, 'Z/9': 1 - p**-n, '(Z/3)^2': p * (1 - p**-n) * (1 - p**(1 - n))}
exact_non = {'Z/3': 1 - p**-n, 'Z/9': 1 - p**-n, '(Z/3)^2': (1 - p**-n) * (1 - p**(1 - n))}
for sym in (True, False):
    acc = {g: 0 for g in exact_sym}
    for _ in range(runs):
        M = rng.integers(0, p**k, size=(n, n))
        if sym: M = np.triu(M) + np.triu(M, 1).T
        for g, c in moments(cok_exponents(M.tolist(), p, k)).items(): acc[g] += c
    ex = exact_sym if sym else exact_non
    print('symmetric' if sym else 'non-symmetric', {g: (round(acc[g] / runs, 3), round(ex[g], 3)) for g in acc})

symmetric {'Z/3': (1.002, 0.999), 'Z/9': (0.97, 0.999), '(Z/3)^2': (2.906, 2.984)}


non-symmetric {'Z/3': (0.988, 0.999), 'Z/9': (0.989, 0.999), '(Z/3)^2': (0.994, 0.995)}


Pairs (simulation, exact). For $\mathbb Z/3$ and $\mathbb Z/9$ the symmetric and non-symmetric moments agree ($\wedge^2$ of a cyclic group is trivial); for $(\mathbb Z/3)^2$ the symmetric moment is $3$ times larger, the factor $|\wedge^2G|=3$. The estimate for $(\mathbb Z/3)^2$ is the noisiest, because it is driven by the rare matrices with $K\ge2$.

**P5.4.** Power series as lists of coefficients.

In [5]:
D = 40
def mul(a, b): return [sum(a[i] * b[j - i] for i in range(j + 1)) for j in range(D + 1)]
def inv_one_minus(e):            # 1/(1 - x^e)
    return [1 if j % e == 0 else 0 for j in range(D + 1)]
lhs = [0] * (D + 1)
for k in range(D + 1):
    if k * (k + 1) // 2 > D: break
    t = [0] * (D + 1); t[k * (k + 1) // 2] = 1
    for j in range(1, k + 1): t = mul(t, inv_one_minus(j))
    lhs = [a + b for a, b in zip(lhs, t)]
distinct = [1] + [0] * D
for i in range(1, D + 1):
    distinct = [distinct[j] + (distinct[j - i] if j >= i else 0) for j in range(D + 1)]
odd = [1] + [0] * D
for i in range(1, D + 1, 2): odd = mul(odd, inv_one_minus(i))
print('sum_k x^(k(k+1)/2)/(x)_k == prod(1+x^i):', lhs == distinct, '   == 1/prod(1-x^(2i-1)):', distinct == odd)
print('first coefficients:', distinct[:15])
for q in (2, 3, 5):
    w = [q**(-k*(k+1)/2) / math.prod(1 - q**-j for j in range(1, k + 1)) for k in range(12)]
    pi0 = math.prod(1 - q**-(2*i - 1) for i in range(1, 60))
    print(f'q={q}: pi_0 = {pi0:.4f} (1/sum = {1/sum(w):.4f}), law', [round(pi0 * x, 4) for x in w[:5]])

sum_k x^(k(k+1)/2)/(x)_k == prod(1+x^i): True    == 1/prod(1-x^(2i-1)): True
first coefficients: [1, 1, 1, 2, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 22]
q=2: pi_0 = 0.4194 (1/sum = 0.4194), law [0.4194, 0.4194, 0.1398, 0.02, 0.0013]
q=3: pi_0 = 0.6390 (1/sum = 0.6390), law [0.639, 0.3195, 0.0399, 0.0015, 0.0]
q=5: pi_0 = 0.7933 (1/sum = 0.7933), law [0.7933, 0.1983, 0.0083, 0.0001, 0.0]


The three series agree up to $x^{40}$ (the coefficients $1,1,1,2,2,3,4,5,6,8,10,\dots$ count partitions into distinct parts), and the normalising constant computed as $1/\sum_k$ matches the product $\prod(1-q^{-(2i-1)})$.

**P5.5.** Same elimination, $3000$ matrices of size $30$ of each kind.

In [6]:
q, n, runs = 3, 30, 3000
tab = {}
for sym in (True, False):
    c = np.zeros(6)
    for _ in range(runs):
        M = rng.integers(0, q, size=(n, n))
        if sym: M = np.triu(M) + np.triu(M, 1).T
        c[min(n - rank_mod(M.tolist(), q), 5)] += 1
    tab[sym] = c / runs
w = [q**(-k*(k+1)/2) / math.prod(1 - q**-j for j in range(1, k + 1)) for k in range(6)]
pis = np.array(w) / sum(w)
c_q = math.prod(1 - q**-j for j in range(1, 80))
pin = [q**(-k*k) * math.prod(1 - q**-j for j in range(k + 1, 80))**2 / c_q for k in range(6)]
for k in range(4):
    print(f'k={k}: symmetric {tab[True][k]:.4f} (limit {pis[k]:.4f})   non-symmetric {tab[False][k]:.4f} (limit {pin[k]:.4f})')

k=0: symmetric 0.6470 (limit 0.6390)   non-symmetric 0.5623 (limit 0.5601)
k=1: symmetric 0.3060 (limit 0.3195)   non-symmetric 0.4140 (limit 0.4201)
k=2: symmetric 0.0450 (limit 0.0399)   non-symmetric 0.0237 (limit 0.0197)
k=3: symmetric 0.0020 (limit 0.0015)   non-symmetric 0.0000 (limit 0.0001)


Symmetric matrices over $\mathbb F_3$ are \emph{less} often singular than general ones ($1-\pi_0\approx0.36$ against $1-\prod_j(1-3^{-j})\approx0.44$), but their large coranks are much more frequent: corank $2$ has limit probability $0.040$ against $0.020$, and corank $3$ has $0.0015$ against $0.0001$. The decay $q^{-k^2/2}$ against $q^{-k^2}$ shows from $k=2$ on; the simulated frequencies are within sampling error of the limits.

**P5.6.** The $3$-rank is $(n-1)$ minus the rank of the reduced Laplacian modulo $3$. For cyclicity, use the exponent: the exponent of $\operatorname{cok}\tilde L$ is the least $e$ with $e\tilde L^{-1}$ integral, that is, the lcm of the denominators of $\tilde L^{-1}$, and the group is cyclic exactly when its exponent equals its order $\det\tilde L$. We compute $\tilde L^{-1}$ exactly with fractions.

In [7]:
def inverse_and_det(A):
    n = len(A); M = [[Fraction(x) for x in r] + [Fraction(int(i == j)) for j in range(n)] for i, r in enumerate(A)]; det = Fraction(1)
    for c in range(n):
        piv = next((i for i in range(c, n) if M[i][c]), None)
        if piv is None: return None, 0
        if piv != c: M[c], M[piv] = M[piv], M[c]; det = -det
        det *= M[c][c]; inv = 1 / M[c][c]; M[c] = [x * inv for x in M[c]]
        for i in range(n):
            if i != c and M[i][c]:
                f = M[i][c]; M[i] = [x - f * y for x, y in zip(M[i], M[c])]
    return [r[n:] for r in M], int(det)

def random_graph_lap(n):
    U = np.triu(rng.random((n, n)) < 0.5, 1).astype(np.int64); Adj = U + U.T
    return (np.diag(Adj.sum(1)) - Adj)[1:, 1:]

for n in (12, 20, 28):
    runs, cyc, conn, r3 = 300, 0, 0, np.zeros(4)
    for _ in range(runs):
        L = random_graph_lap(n).tolist()
        Linv, det = inverse_and_det(L)
        if det == 0: continue               # disconnected graph
        conn += 1
        e = math.lcm(*[x.denominator for r in Linv for x in r])
        cyc += (e == det)
        r3[min(n - 1 - rank_mod(L, 3), 3)] += 1
    print(f'n={n}: connected {conn}/{runs}, cyclic proportion {cyc/conn:.3f}, 3-rank law {np.round(r3/conn, 3)}')
print('Wood bound for the cyclic proportion: prod zeta(2j+1)^-1 =', round(math.prod(1/sum(m**-(2*j + 1) for m in range(1, 4000)) for j in range(1, 30)), 4))

n=12: connected 299/300, cyclic proportion 0.789, 3-rank law [0.635 0.331 0.03  0.003]


n=20: connected 300/300, cyclic proportion 0.797, 3-rank law [0.637 0.33  0.03  0.003]


n=28: connected 300/300, cyclic proportion 0.823, 3-rank law [0.683 0.29  0.023 0.003]
Wood bound for the cyclic proportion: prod zeta(2j+1)^-1 = 0.7935


The cyclic proportion is already between $0.79$ and $0.82$ for small graphs, consistent with the limit $\prod_j\zeta(2j+1)^{-1}\approx0.7935$ of Remark 4.5 (an upper bound, conjecturally the truth), with an error of about $0.025$ from $300$ samples. The $3$-rank law is close to $\pi=(0.639,0.320,0.040,0.0015)$ of Theorem 4.4 for $q=3$, although the Laplacian is nothing like a uniform matrix: that is Wood's universality.